# QALF Colab Notebook

This notebook is Colab-ready. It mounts Google Drive, finds or clones the `EXPLLM` repository, imports the live `qalf` package, and can resume training from a Drive checkpoint.

Current checkpoint target:
`/content/drive/MyDrive/EXPLLM/runs/qalf_v3/checkpoint_epoch_100.pt`

In [ ]:
# Colab/bootstrap cell. Safe to re-run.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/TheFausap/EXPLLM.git"
DRIVE_PROJECT = Path("/content/drive/MyDrive/EXPLLM")
LOCAL_REPO = Path("/content/EXPLLM")

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)

def has_qalf_repo(path: Path) -> bool:
    return (path / "qalf" / "model.py").exists() and (path / "qalf" / "train.py").exists()

if has_qalf_repo(DRIVE_PROJECT):
    ROOT = DRIVE_PROJECT
elif IN_COLAB:
    if not has_qalf_repo(LOCAL_REPO):
        if LOCAL_REPO.exists():
            raise RuntimeError(f"{LOCAL_REPO} exists but does not look like EXPLLM. Rename it or set ROOT manually.")
        subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO)], check=True)
    ROOT = LOCAL_REPO
else:
    ROOT = Path.cwd()
    if ROOT.name == "qalf":
        ROOT = ROOT.parent
    if not has_qalf_repo(ROOT):
        raise RuntimeError("Cannot find the EXPLLM repository. Set ROOT to the repo path manually.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("IN_COLAB:", IN_COLAB)
print("ROOT:", ROOT)
print("cwd:", Path.cwd())

## 1. Environment And Paths

`DATA` is the dataset path you already edited. `CHECKPOINT` points to the epoch-100 Drive checkpoint. Change `RESUME_TO_EPOCH` before running the resume cell if you want a different target epoch.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import torch

DATA = Path('/content/drive/MyDrive/EXPLLM/data/dolly_qalf.jsonl')
CHECKPOINT = Path("/content/drive/MyDrive/EXPLLM/runs/qalf_v3/checkpoint_epoch_100.pt")
RUN_DIR = CHECKPOINT.parent

# Resume controls. --epochs is the final epoch number, not additional epochs.
RESUME_TO_EPOCH = 120
RESUME_BATCH_SIZE = 512
RESUME_LR = 0.004
RESUME_LR_SCHEDULE = "constant"
RESUME_SAVE_EVERY = 5
RESUME_MAX_WINDOWS = None       # Example: 2_000_000 for a faster Colab continuation.
RESUME_BIGRAM_DEVICE = None     # Example: "cpu" to save GPU VRAM at some speed cost.

# Prior/regularisation controls used when rebuilding memory from DATA during resume.
RESUME_TRIGRAM_TOP_K = 96
RESUME_TRIGRAM_MIN_COUNT = 2
RESUME_TRIGRAM_STRENGTH = 1.0
RESUME_BIGRAM_STRENGTH = 0.35
RESUME_ENTROPY_WEIGHT = 0.02
RESUME_COMPONENT_DIVERSITY_WEIGHT = 0.1
RESUME_COMPONENT_DIVERSITY_TARGET = 0.05
RESUME_COMPONENT_TEMPERATURE = 2.0
RESUME_COMPONENT_MIN_WEIGHT = 0.08
RESUME_LOG = RUN_DIR / "resume_from_epoch_100.jsonl"

from qalf.data import (
    build_tokenizer,
    encode_examples,
    make_windows,
    read_jsonl,
    relation_counts,
    trigram_counts,
)
from qalf.model import QALFConfig, QALFModel, cross_entropy_with_l2, device_for_training, load_checkpoint

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("DATA:", DATA, "exists=", DATA.exists())
print("CHECKPOINT:", CHECKPOINT, "exists=", CHECKPOINT.exists())
print("RUN_DIR:", RUN_DIR)

## 2. Optional Notebook Smoke Data

This section builds a small notebook-side tokenizer/model view from `DATA` for quick sanity checks. If your goal is only to continue the Drive checkpoint, you can skip sections 2-4 and run section 7 directly after the environment/path cell.

In [ ]:
examples = read_jsonl(DATA)
tokenizer = build_tokenizer(examples, vocab_size=512)
encoded = encode_examples(tokenizer, examples)

context_size = 24
contexts, prev2_tokens, prev_tokens, targets = make_windows(
    encoded,
    context_size=context_size,
    pad_id=tokenizer.pad_id,
    include_prev2=True,
)
bigram = relation_counts(encoded, len(tokenizer.vocab))
trigram = trigram_counts(encoded, len(tokenizer.vocab), top_k=16, min_count=1)

print("examples:", len(examples))
print("vocab:", len(tokenizer.vocab))
print("windows:", int(targets.numel()))
print("trigram contexts:", int(trigram["keys"].numel()))

## 3. Instantiate QALF-Mixed

A useful sanity check is `purity_mean`: with multiple context components it should usually be below 1.0, which means the context is being represented as a mixed density state rather than a single pure vector.

In [ ]:
device = device_for_training("auto")
config = QALFConfig(
    vocab_size=len(tokenizer.vocab),
    dimension=64,
    context_size=context_size,
    num_relations=4,
    num_components=4,
    bigram_strength=0.35,
    trigram_strength=0.75,
    pad_id=tokenizer.pad_id,
)
model = QALFModel(config, bigram_logits=bigram, trigram_prior=trigram).to(device)

batch = contexts[:8].to(device)
logits = model(batch, prev_tokens[:8].to(device), prev2_tokens[:8].to(device))
print("logits shape:", tuple(logits.shape))
print(model.diagnostics(batch))

## 4. Tiny Smoke Training

This is only a functionality check. It verifies forward/backward, the mixed-state diagnostics, and generation. Real experiments should use the CLI commands below so checkpointing and job logs are captured.

In [ ]:
dataset = torch.utils.data.TensorDataset(contexts, prev2_tokens, prev_tokens, targets)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.train()
for step, (bc, bp2, bp, bt) in enumerate(loader, start=1):
    bc, bp2, bp, bt = bc.to(device), bp2.to(device), bp.to(device), bt.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(bc, bp, bp2)
    loss = cross_entropy_with_l2(model, logits, bt, entropy_weight=0.02)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 5 == 0 or step == 1:
        print({"step": step, "loss": float(loss.detach().cpu())})
    if step >= 20:
        break

model.eval()
with torch.no_grad():
    print(model.diagnostics(contexts[:8].to(device)))

In [ ]:
prompt = "What is QALF?"
print("prompt:", prompt)
print("reply:", model.generate(tokenizer, prompt, max_new_tokens=48, temperature=0.0, top_k=18, seed=7, device=device))

## 5. Prepare A Bigger Dataset

Example TinyStories preparation with a JSONL job log:

```bash
conda run -n EXPLLM python -m qalf.prepare_text \
  --source tinystories-train \
  --out data/tinystories_train_100k_qalf.jsonl \
  --max-examples 100000 \
  --prompt-tokens 24 \
  --reply-tokens 128 \
  --log-file runs/qalf_mixed_dgx/prep.jsonl
```

You can also pass a local file or JSONL dataset via `--source file` or `--source jsonl`.

## 6. Recommended QALF-Mixed Training Run

This is the current comparison-scale command for a larger GPU box. It uses constant LR because the first warmup-cosine run decayed too early for this underpowered objective. The CLI still supports `--lr-schedule warmup-cosine` for controlled tests.

```bash
conda run -n EXPLLM python -m qalf.train \
  --data data/tinystories_train_100k_qalf.jsonl \
  --out runs/qalf_mixed_compare \
  --device cuda \
  --dimension 512 \
  --context-size 128 \
  --components 6 \
  --vocab-size 32000 \
  --epochs 8 \
  --batch-size 2048 \
  --lr 0.004 \
  --lr-schedule constant \
  --max-windows 10000000 \
  --relations 12 \
  --trigram-top-k 96 \
  --trigram-min-count 2 \
  --trigram-strength 1.0 \
  --bigram-strength 0.35 \
  --entropy-weight 0.02 \
  --component-diversity-weight 0.1 \
  --component-diversity-target 0.05 \
  --component-temperature 2.0 \
  --component-min-weight 0.08 \
  --attractor-limit 2000 \
  --save-every 2 \
  --log-every 1 \
  --log-file runs/qalf_mixed_compare/train.jsonl
```

Useful extra flags:
- `--bigram-decay-epochs N` and `--trigram-decay-epochs N` to make the symbolic priors fade during training.
- `--bigram-device cpu` to save GPU memory if the vocabulary is large.
- `--resume runs/.../checkpoint_epoch_N.pt` to continue a run.
- `--reset-optimizer` when resuming across architecture changes.
- `--reset-lr-schedule` when resuming and intentionally restarting the LR schedule.

## 7. Resume From The Google Drive Checkpoint

This cell resumes training from:
`/content/drive/MyDrive/EXPLLM/runs/qalf_v3/checkpoint_epoch_100.pt`

The trainer reloads the checkpoint model/tokenizer, rebuilds bigram/trigram priors from `DATA`, restores optimizer state when compatible, and writes a JSONL job log to Drive. `--epochs` is the final target epoch, not the number of additional epochs.

In [ ]:
resume_cmd = [
    sys.executable, "-m", "qalf.train",
    "--data", str(DATA),
    "--out", str(RUN_DIR),
    "--resume", str(CHECKPOINT),
    "--epochs", str(RESUME_TO_EPOCH),
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
    "--batch-size", str(RESUME_BATCH_SIZE),
    "--lr", str(RESUME_LR),
    "--lr-schedule", RESUME_LR_SCHEDULE,
    "--trigram-top-k", str(RESUME_TRIGRAM_TOP_K),
    "--trigram-min-count", str(RESUME_TRIGRAM_MIN_COUNT),
    "--trigram-strength", str(RESUME_TRIGRAM_STRENGTH),
    "--bigram-strength", str(RESUME_BIGRAM_STRENGTH),
    "--entropy-weight", str(RESUME_ENTROPY_WEIGHT),
    "--component-diversity-weight", str(RESUME_COMPONENT_DIVERSITY_WEIGHT),
    "--component-diversity-target", str(RESUME_COMPONENT_DIVERSITY_TARGET),
    "--component-temperature", str(RESUME_COMPONENT_TEMPERATURE),
    "--component-min-weight", str(RESUME_COMPONENT_MIN_WEIGHT),
    "--save-every", str(RESUME_SAVE_EVERY),
    "--log-every", "1",
    "--log-file", str(RESUME_LOG),
]
if RESUME_MAX_WINDOWS is not None:
    resume_cmd.extend(["--max-windows", str(RESUME_MAX_WINDOWS)])
if RESUME_BIGRAM_DEVICE is not None:
    resume_cmd.extend(["--bigram-device", str(RESUME_BIGRAM_DEVICE)])

print(" ".join(resume_cmd))
subprocess.run(resume_cmd, check=True)

In [ ]:
# Inspect the latest resume log records.
if RESUME_LOG.exists():
    for line in RESUME_LOG.read_text(encoding="utf-8").strip().splitlines()[-10:]:
        print(json.loads(line))
else:
    print("Resume log not found yet:", RESUME_LOG)

In [ ]:
# Load the checkpoint/model for a quick generation check.
# If you just resumed, change CHECKPOINT_TO_LOAD to RUN_DIR / "model.pt" or a newer checkpoint_epoch_N.pt.
CHECKPOINT_TO_LOAD = CHECKPOINT
model, checkpoint_tokenizer, metadata = load_checkpoint(CHECKPOINT_TO_LOAD, map_location="cpu")
device = device_for_training("auto")
model = model.to(device).eval()
print("loaded:", CHECKPOINT_TO_LOAD)
print("config:", model.config)
print("metadata keys:", sorted(metadata.keys()))

prompt = "Tell me a small story about a blue robot."
with torch.no_grad():
    print(model.generate(checkpoint_tokenizer, prompt, max_new_tokens=80, temperature=0.8, top_k=24, seed=7, device=device))

## 8. Read Job Logs

Training and preparation logs are JSONL files. This helper prints the latest records from a run directory.

In [ ]:
LOG = ROOT / "runs" / "qalf_mixed_compare" / "train.jsonl"
if LOG.exists():
    lines = LOG.read_text(encoding="utf-8").strip().splitlines()
    for line in lines[-5:]:
        print(json.loads(line))
else:
    print("No log found yet:", LOG)

## 9. Notebook Drift Check

This notebook should stay small. If a future code change adds arguments or model fields, update the import/demo cells and the CLI recipes here, but keep the implementation in the Python package.